In [1]:
# installing needed dependencies on colab
!pip install unsloth torch trl evaluate transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Load necessary libraries for GRPO
import unsloth

from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments,GenerationConfig
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel

import torch
from datasets import load_dataset

from evaluate import load
from tqdm import tqdm
import json


print("GRPO libraries loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GRPO libraries loaded successfully!


In [4]:
max_seq_length = 768 # increase if task needs longer outputs
lora_rank = 4 # higher LoRA rank -> better approximation but more VRAM needed

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    offload_embedding=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    # 'target_modules' are the modules of our model we wanna apply the Low Rank Adaptation on
    # we can check the name of the available modules in our model simply by print(model)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",     # big memory saver
    random_state = 3407,
)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [36]:
messages = [
    {
        "role" :  "system", # <--- define the high-level directives for the model
        "content" : "You're a very smart guy."
    },
    {
        "role" :  "user",   # <--- user query
        "content" : """You are playing a game with another agent A. You must choose either action action3 or action action4. Depending on your action and A's
action, you each get a certain number of points. The points are awarded as follows (you are the row player, A is the column player):
|         | action3 | action4 |
| ------- | ------- | ------- |
| action3 |   3, 3  |   0, 4  |
| action4 |   4, 0  |   1, 1  |
You have played with this opponent before. Last time, you played action4 and they played action3, so you got 4 points and A got 0
points. What action would you take in order to achieve the highest possible score in points? Your answer:"""
    }
]

# apply_chat_template() is fundamental for an instruction model.
# an instruction model without apply_chat_template() is just like a base model!!
inputs = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(inputs)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Feb 2026

You're a very smart guy.<|eot_id|><|start_header_id|>user<|end_header_id|>

You are playing a game with another agent A. You must choose either action action3 or action action4. Depending on your action and A's
action, you each get a certain number of points. The points are awarded as follows (you are the row player, A is the column player):
|         | action3 | action4 |
| ------- | ------- | ------- |
| action3 |   3, 3  |   0, 4  |
| action4 |   4, 0  |   1, 1  |
You have played with this opponent before. Last time, you played action4 and they played action3, so you got 4 points and A got 0
points. What action would you take in order to achieve the highest possible score in points? Your answer:<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [37]:
inputs = tokenizer.apply_chat_template(messages,
                                       tokenize=True,
                                       add_generation_prompt=True,
                                       return_tensors="pt")
inputs = inputs.to(model.device)

output_ids = model.generate(
    inputs,
    max_new_tokens=256,
    pad_token_id=tokenizer.pad_token_id
)
output = tokenizer.decode(output_ids[0][inputs.shape[-1]:],
                          skip_special_tokens=True)

print("\nOUTPUT:", output)


OUTPUT: Given the points awarded in the game, I can analyze the possible outcomes of each action.

Since I have played with this opponent before, I can recall the outcome of our previous game, where I played action4 and they played action3, resulting in 4 points for me and 0 points for them.

To achieve the highest possible score, I need to maximize my points. Based on the points awarded, I can see that playing action4 would result in 4 points, while playing action3 would result in 3 points.

Considering our previous game, I can infer that my opponent is likely to play action3 if I play action4. Therefore, playing action4 would give me the highest possible score.

So, my action would be: action4.
